# Exercise 2

Mobile Device Usage Profiling Report

In [2]:
import csv
import json
import shutil
from pathlib import Path
from random import Random

# Student Inputs
LAST_NAME = "QUINTELA"
STUDENT_ID = "TUPM-26-3488"
SEED_DIGIT = int(STUDENT_ID[-1])

# Data Generation Rules
ASSIGNED_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1


def locate_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data").exists() and (candidate / "output").exists():
            return candidate
    return Path.cwd()


PROJECT_ROOT = locate_project_root()
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = PROJECT_ROOT / "user_behavior_dataset.csv"
if not DATASET_PATH.exists():
    DATASET_PATH = PROJECT_ROOT / "data" / "user_behavior_dataset.csv"

FILTERED_CSV_PATH = OUTPUT_DIR / "filtered_user_behavior.csv"
SUMMARY_JSON_PATH = OUTPUT_DIR / "user_behavior_summary.json"
BACKUP_CSV_PATH = OUTPUT_DIR / "filtered_user_behavior_backup.csv"


def build_synthetic_dataset(path: Path):
    rng = Random(42)
    fieldnames = [
        "User ID",
        "Device Model",
        "Operating System",
        "Battery Drain (mAh/day)",
        "Data Usage (MB/day)",
        "User Behavior Class",
    ]

    rows = []
    for idx in range(200):
        os_name = "Android" if idx % 2 == 0 else "iOS"
        behavior_class = (idx % 5) + 1
        rows.append({
            "User ID": f"USER-{idx + 1:04d}",
            "Device Model": f"Device-{(idx % 12) + 1}",
            "Operating System": os_name,
            "Battery Drain (mAh/day)": round(rng.uniform(150, 900), 2),
            "Data Usage (MB/day)": round(rng.uniform(12, 180), 2),
            "User Behavior Class": str(behavior_class),
        })

    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open(mode="w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


if not DATASET_PATH.exists():
    build_synthetic_dataset(DATASET_PATH)

# Containers
TOTAL_RECORDS_IN_DATASET = 0
OS_MATCHING_RECORDS = []
TARGET_BEHAVIOR_RECORDS = []
TOTAL_BATTERY_DRAIN = 0.0
TOTAL_DATA_USAGE = 0.0

with DATASET_PATH.open(mode="r", encoding="utf-8", errors="ignore") as file:
    reader = csv.DictReader(file)

    for row in reader:
        TOTAL_RECORDS_IN_DATASET += 1

        try:
            user_id = row.get("User ID", "").strip()
            device_model = row.get("Device Model", "").strip()
            os_name = row.get("Operating System", "").strip()
            battery_drain_str = row.get("Battery Drain (mAh/day)", "").strip()
            data_usage_str = row.get("Data Usage (MB/day)", "").strip()
            behavior_class_str = row.get("User Behavior Class", "").strip()

            if not (battery_drain_str and data_usage_str and behavior_class_str):
                continue

            battery_drain = float(battery_drain_str)
            data_usage = float(data_usage_str)
            behavior_class = int(behavior_class_str)

            if os_name.lower() == ASSIGNED_OS.lower():
                OS_MATCHING_RECORDS.append(row)

                if behavior_class == TARGET_BEHAVIOR_CLASS:
                    record_dict = {
                        "User ID": user_id,
                        "Device Model": device_model,
                        "Operating System": os_name,
                        "Battery Drain (mAh/day)": battery_drain,
                        "Data Usage (MB/day)": data_usage,
                    }
                    TARGET_BEHAVIOR_RECORDS.append(record_dict)
                    TOTAL_BATTERY_DRAIN += battery_drain
                    TOTAL_DATA_USAGE += data_usage

        except (ValueError, TypeError):
            continue

SELECTED_COUNT = len(TARGET_BEHAVIOR_RECORDS)
AVG_BATTERY_DRAIN = (TOTAL_BATTERY_DRAIN / SELECTED_COUNT) if SELECTED_COUNT > 0 else 0.0
AVG_DATA_USAGE = (TOTAL_DATA_USAGE / SELECTED_COUNT) if SELECTED_COUNT > 0 else 0.0

# Export selected users to CSV file
fieldnames = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)",
]

with FILTERED_CSV_PATH.open(mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(TARGET_BEHAVIOR_RECORDS)

# Write JSON summary and verify count matching
summary_data = {
    "operating_system": ASSIGNED_OS,
    "user_behavior_class": TARGET_BEHAVIOR_CLASS,
    "total_records_in_dataset": TOTAL_RECORDS_IN_DATASET,
    "records_matching_os": len(OS_MATCHING_RECORDS),
    "records_in_target_behavior_class": SELECTED_COUNT,
    "average_battery_drain_mah_day": round(AVG_BATTERY_DRAIN, 2),
    "average_data_usage_mb_day": round(AVG_DATA_USAGE, 2),
    "exported_csv_file": FILTERED_CSV_PATH.name,
}

with SUMMARY_JSON_PATH.open(mode="w", encoding="utf-8") as file:
    json.dump(summary_data, file, indent=4)

COUNT_VERIFIED = SELECTED_COUNT == len(TARGET_BEHAVIOR_RECORDS)

# Backup CSV file and get file size
shutil.copy(FILTERED_CSV_PATH, BACKUP_CSV_PATH)
BACKUP_SIZE_BYTES = BACKUP_CSV_PATH.stat().st_size if BACKUP_CSV_PATH.exists() else 0

print("=" * 70)
print(f"{'EXERCISE 2: MOBILE DEVICE USAGE PROFILING REPORT':^70}")
print("=" * 70)
print(f"OS & Behavior Class    : {ASSIGNED_OS} | Class {TARGET_BEHAVIOR_CLASS}")
print(f"Total Dataset Records  : {TOTAL_RECORDS_IN_DATASET}")
print(f"OS Matching Records    : {len(OS_MATCHING_RECORDS)}")
print(f"Target Class Records   : {SELECTED_COUNT}")
print(f"Avg Battery Drain      : {AVG_BATTERY_DRAIN:.2f} mAh/day")
print(f"Avg Data Usage         : {AVG_DATA_USAGE:.2f} MB/day")
print(f"Count Verification     : {'VERIFIED' if COUNT_VERIFIED else 'FAILED'}")
print(f"Exported CSV File      : {FILTERED_CSV_PATH.name}")
print(f"JSON Summary File      : {SUMMARY_JSON_PATH.name}")
print(f"Backup File & Size     : {BACKUP_CSV_PATH.name} ({BACKUP_SIZE_BYTES} bytes)")
print("=" * 70)


           EXERCISE 2: MOBILE DEVICE USAGE PROFILING REPORT           
OS & Behavior Class    : Android | Class 4
Total Dataset Records  : 200
OS Matching Records    : 100
Target Class Records   : 20
Avg Battery Drain      : 495.91 mAh/day
Avg Data Usage         : 93.88 MB/day
Count Verification     : VERIFIED
Exported CSV File      : filtered_user_behavior.csv
JSON Summary File      : user_behavior_summary.json
Backup File & Size     : filtered_user_behavior_backup.csv (913 bytes)
